In [17]:
import tensorflow as tf
import os
import filetype
import cv2
import random
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import glob
import numpy as np
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
import re
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (classification_report, ConfusionMatrixDisplay, roc_auc_score,
                             precision_recall_curve, RocCurveDisplay, PrecisionRecallDisplay)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [18]:
data_dir = 'data'
imageTypes = ["jpg", "jpeg", "png", "bmp"]

In [19]:
'''
for data_type in os.listdir(data_dir):
    for image_class in os.listdir(data_dir): 
        for image in os.listdir(os.path.join(data_dir, image_class)):
            image_path = os.path.join(data_dir, image_class, image)
            try: 
                img = cv2.imread(image_path)
                kind = filetype.guess(image_path)
                if kind is None or kind.extension not in imageTypes:
                    print(f'Image {image_path} is of type {kind}. Should be in {imageTypes}')
                    os.remove(image_path)
            except Exception as e: 
                print('Issue with image {}'.format(image_path))
                os.remove(image_path)
'''

"\nfor data_type in os.listdir(data_dir):\n    for image_class in os.listdir(data_dir): \n        for image in os.listdir(os.path.join(data_dir, image_class)):\n            image_path = os.path.join(data_dir, image_class, image)\n            try: \n                img = cv2.imread(image_path)\n                kind = filetype.guess(image_path)\n                if kind is None or kind.extension not in imageTypes:\n                    print(f'Image {image_path} is of type {kind}. Should be in {imageTypes}')\n                    os.remove(image_path)\n            except Exception as e: \n                print('Issue with image {}'.format(image_path))\n                os.remove(image_path)\n"

In [20]:
train_normal_dir = os.path.join(data_dir, 'train/NORMAL')
train_pneu_dir = os.path.join(data_dir, 'train/PNEUMONIA')

test_normal_dir = os.path.join(data_dir, 'test/NORMAL')
test_pneu_dir = os.path.join(data_dir, 'test/PNEUMONIA')

val_normal_dir = os.path.join(data_dir, 'val/NORMAL')
val_pneu_dir = os.path.join(data_dir, 'val/PNEUMONIA')

In [21]:
train_list = []
test_list = []
val_list = []

In [22]:
def get_all_images(directory, imageTypes):
    files = []
    for ext in imageTypes:
        files.extend(glob.glob(os.path.join(directory, f'*.{ext}')))
    return files

train_normal_cases = get_all_images(train_normal_dir, imageTypes)
train_pneu_cases = get_all_images(train_pneu_dir, imageTypes)

test_normal_cases = get_all_images(test_normal_dir, imageTypes)
test_pneu_cases = get_all_images(test_pneu_dir, imageTypes)

val_normal_cases = get_all_images(val_normal_dir, imageTypes)
val_pneu_cases = get_all_images(val_pneu_dir, imageTypes)

In [23]:
train_normal_cases = [x.replace('\\', '/') for x in train_normal_cases]
train_pneu_cases = [x.replace('\\', '/') for x in train_pneu_cases]
test_normal_cases = [x.replace('\\', '/') for x in test_normal_cases]
test_pneu_cases = [x.replace('\\', '/') for x in test_pneu_cases]
val_normal_cases = [x.replace('\\', '/') for x in val_normal_cases]
val_pneu_cases = [x.replace('\\', '/') for x in val_pneu_cases]

In [24]:
for img in train_normal_cases:
    train_list.append([img, 0])
    
for img in train_pneu_cases:
    train_list.append([img, 1])
    
for img in test_normal_cases:
    test_list.append([img, 0])
    
for img in test_pneu_cases:
    test_list.append([img, 1])
    
for img in val_normal_cases:
    val_list.append([img, 0])
    
for img in val_pneu_cases:
    val_list.append([img, 1])

In [25]:
random.shuffle(train_list)

In [ ]:
train_df = pd.DataFrame(train_list, columns=['image', 'label'])
test_df = pd.DataFrame(test_list, columns=['image', 'label'])
val_df = pd.DataFrame(val_list, columns=['image', 'label'])

### Validation split

The dataset's own `val/` folder has only 16 images, which is too few to guide training, so V2 originally monitored the **test set** during training and its test score was optimistic.

Instead, the 16 images are folded back into the training pool and ~15% is held out as a proper validation set. The split is:
- **stratified**, so validation keeps the same NORMAL/PNEUMONIA ratio as training
- **grouped by patient**, since many patients have several X-rays (`person123_bacteria_1.jpeg`, `person123_bacteria_2.jpeg`, ...). Otherwise near-duplicate images of the same patient would leak between train and validation.

The test set is not touched until the final evaluation.

In [ ]:
def patient_id(img_path):
    # person123_bacteria_456.jpeg -> person123, IM-0115-0001.jpeg -> IM-0115, NORMAL2-IM-0123-0001.jpeg -> NORMAL2-IM-0123
    return re.match(r'(person\d+|(?:NORMAL2-)?IM-\d+)', os.path.basename(img_path)).group(1)

pool_df = pd.concat([train_df, val_df], ignore_index=True)
groups = pool_df['image'].map(patient_id)

# 7 folds -> one fold is ~14% of the data
splitter = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
train_idx, val_idx = next(splitter.split(pool_df, pool_df['label'], groups))
train_df = pool_df.iloc[train_idx].reset_index(drop=True)
val_df = pool_df.iloc[val_idx].reset_index(drop=True)

assert not set(groups.iloc[train_idx]) & set(groups.iloc[val_idx])
print(f'Train: {len(train_df)} images, Validation: {len(val_df)} images')
print('Validation class balance:', val_df['label'].value_counts().sort_index().to_dict())

In [ ]:
plt.figure(figsize=(20,5))
labels = ["NORMAL", "PNEUMONIA"]

for i, (name, df) in enumerate([("Training Set", train_df), ("Test Set", test_df), ("Validation Set", val_df)]):
    plt.subplot(1, 3, i+1)
    plt.bar(labels, df['label'].value_counts().sort_index().values)
    plt.title(name)

plt.show()

In [ ]:
def process_data(img_path):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (196, 196))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img = img/255.0
    img = np.reshape(img, (196,196,1))
    return img

def compose_dataset(df):
    data = []
    labels = []
    for img_path, label in df.values:
        data.append(process_data(img_path))
        labels.append(label)
        
    return np.array(data), np.array(labels)

X_train, y_train = compose_dataset(train_df)
X_test, y_test = compose_dataset(test_df)
X_val, y_val = compose_dataset(val_df)

In [ ]:
print('Train data shape: {}, Labels shape: {}'.format(X_train.shape, y_train.shape))
print('Test data shape: {}, Labels shape: {}'.format(X_test.shape, y_test.shape))
print('Validation data shape: {}, Labels shape: {}'.format(X_val.shape, y_val.shape))


In [ ]:
plt.figure(figsize=(20,8))
for i,img_path in enumerate(train_df[train_df['label'] == 1][0:4]['image']):
    plt.subplot(2,4,i+1)
    plt.axis('off')
    img = plt.imread(img_path)
    plt.imshow(img, cmap='gray')
    plt.title('Pneumonia')
    
for i,img_path in enumerate(train_df[train_df['label'] == 0][0:4]['image']):
    plt.subplot(2,4,4+i+1)
    plt.axis('off')
    img = plt.imread(img_path)
    plt.imshow(img, cmap='gray')
    plt.title('Normal')


In [ ]:
y_train_cat = to_categorical(y_train, num_classes=2)
y_test_cat = to_categorical(y_test, num_classes=2)
y_val_cat = to_categorical(y_val, num_classes=2)


In [ ]:
model = Sequential()

model.add(Conv2D(filters=8, kernel_size=(7,7), padding='same', activation='relu', input_shape=(196, 196, 1)))
model.add(Conv2D(filters=8, kernel_size=(7,7), padding='same', activation='relu'))
model.add(MaxPooling2D(pool_size=(3,3)))

model.add(Conv2D(filters=16, kernel_size=(5,5), padding='same', activation='relu'))
model.add(Conv2D(filters=16, kernel_size=(5,5), padding='same', activation='relu'))
model.add(MaxPooling2D(pool_size=(3,3)))

model.add(Conv2D(filters=32, kernel_size=(3,3), padding='same', activation='relu'))
model.add(Conv2D(filters=32, kernel_size=(3,3), padding='same', activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))

model.add(Conv2D(filters=64, kernel_size=(3,3), padding='same', activation='relu'))
model.add(Conv2D(filters=64, kernel_size=(3,3), padding='same', activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))

model.add(Conv2D(filters=128, kernel_size=(3,3), padding='same', activation='relu'))
model.add(Conv2D(filters=128, kernel_size=(3,3), padding='same', activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))

model.add(Flatten())

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(2, activation='softmax'))

'''
optimizer = Adam(lr=0.0001, decay=1e-5)
model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])
'''

model.summary()

optimizer = Adam(learning_rate=0.0001, decay=1e-5)
model.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])


callback = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)
history = model.fit(X_train, y_train_cat, validation_data=(X_val, y_val_cat), batch_size=4, epochs=50, verbose=1, callbacks=[callback], class_weight={0:6.0, 1:0.5})

In [ ]:
from datetime import date

os.makedirs('models', exist_ok=True)
model.save(os.path.join('models', f'v2_cnn_{date.today().isoformat()}.keras'))

## Evaluation

Missing a pneumonia case (false negative) is far worse than a false alarm, so accuracy alone is misleading. This section reports:
- a confusion matrix and per-class precision / recall / F1
- ROC-AUC and the ROC and precision-recall curves
- a **tuned decision threshold** that keeps pneumonia recall at or above a target

The threshold is chosen on the **validation set**, then applied to the test set once, so the test score stays unbiased.

In [ ]:
class_names = ['NORMAL', 'PNEUMONIA']

val_probs = model.predict(X_val, verbose=0)[:, 1]
test_probs = model.predict(X_test, verbose=0)[:, 1]

def evaluate(threshold, title):
    y_pred = (test_probs >= threshold).astype(int)
    print(f'{title} (threshold = {threshold:.3f})')
    print(classification_report(y_test, y_pred, target_names=class_names, digits=3))
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=class_names, cmap='Blues')
    plt.title(title)
    plt.show()

In [ ]:
evaluate(0.5, 'Default threshold')

In [ ]:
print(f'Test ROC-AUC: {roc_auc_score(y_test, test_probs):.3f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
RocCurveDisplay.from_predictions(y_test, test_probs, name='V2 CNN', ax=axes[0])
axes[0].plot([0, 1], [0, 1], 'k--', label='Chance')
axes[0].legend()
axes[0].set_title('ROC curve (test)')
PrecisionRecallDisplay.from_predictions(y_test, test_probs, name='V2 CNN', ax=axes[1])
axes[1].set_title('Precision-recall curve (test)')
plt.show()

### Threshold tuning

Pick the **highest** threshold whose recall on the validation set is still at least `TARGET_RECALL`. A higher threshold means fewer false alarms, so this gives the best precision that still meets the recall target.

In [ ]:
TARGET_RECALL = 0.95

precision, recall, thresholds = precision_recall_curve(y_val, val_probs)
# recall[i] and precision[i] belong to thresholds[i]; the final entries have no threshold
meets_target = recall[:-1] >= TARGET_RECALL
best_threshold = thresholds[meets_target].max()
i = np.where(thresholds == best_threshold)[0][0]
print(f'Chosen threshold: {best_threshold:.3f} '
      f'(validation recall {recall[i]:.3f}, precision {precision[i]:.3f})')

evaluate(best_threshold, f'Tuned threshold (target recall {TARGET_RECALL})')